# Réadmission à 30 jours — patients diabétiques (cas n°12)
Notebook d'accompagnement : mêmes fonctions que l'application (`core/`), exécutées pas à pas. Lancez-le depuis la racine du dépôt (ou `notebooks/`). Aucune sortie n'est pré-enregistrée : exécutez-le pour obtenir vos résultats.

In [ ]:
import os, sys
if not os.path.exists('core'): sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, shap
from core import data as D, models as M, evaluation as EV, explain as EX
from core.prep import FeatureSpace, split_data

## 1. Données — téléchargement automatique (UCI #296) et nettoyage

In [ ]:
raw = D.fetch_diabetes()
print(raw.shape)
raw.head()

In [ ]:
X, y, groups, info = D.prepare_diabetes(raw)   # décès/soins palliatifs retirés, 1 rencontre par patient
print(info['n_raw'], '->', info['n_final'], '| taux positif :', round(y.mean(), 4))
raw['readmitted'].value_counts()

In [ ]:
for c in ['discharge', 'diag_1', 'A1Cresult']:
    print(y.groupby(X[c]).mean().sort_values(ascending=False).round(3), '\n')

## 2. Préparation

In [ ]:
Xtr, Xte, ytr, yte = split_data(X, y, test_size=0.25, seed=42, groups=groups)   # découpage par patient
Xtr = Xtr.sample(min(40000, len(Xtr)), random_state=42); ytr = ytr.loc[Xtr.index]
fs = FeatureSpace().fit(Xtr); A, B = fs.transform(Xtr), fs.transform(Xte)
A.shape, B.shape

## 3. Modèles (avec optimisation des hyperparamètres)

In [ ]:
res = {}
for n in M.MODEL_NAMES:
    r = M.train(n, A, ytr, tune=True, n_iter=6)
    r['proba'] = r['model'].predict_proba(B)[:, 1]
    res[n] = r
T = EV.comparison_table(res, yte); T

## 4. Évaluation

In [ ]:
EV.roc_fig(res, yte).show(); EV.pr_fig(res, yte).show()

In [ ]:
best = T.iloc[0]['Modèle']; proba = res[best]['proba']
thr = EV.best_threshold(yte, proba)
print(best, EV.metrics_at(yte, proba, thr))

## 5. Explications globales

In [ ]:
m = res[best]['model']; rng = np.random.default_rng(0)
bg = A[rng.choice(len(A), 100, replace=False)]; idx = rng.choice(len(B), 1000, replace=False)
ex = EX.make_shap_explainer(best, m, bg); exp = EX.shap_explain(best, ex, B[idx], fs.to_display(B[idx]), fs.names)
shap.plots.beeswarm(exp, max_display=15)

In [ ]:
EX.global_by_variable(exp.values, fs.meta).head(10)

## 6. Explication d'un individu : SHAP, LIME, phrase de synthèse

In [ ]:
from core import service as SV
E = SV.Explainer(SV.make_bundle(fs, m, best, thr, Xtr, A, B, with_global=False))
i = int(np.argmax(proba))   # patient au risque le plus élevé du jeu de test
out = E.explain({k: v for k, v in Xte.iloc[i].items() if not pd.isna(v)}, thr)
print(out['sentence']); print(out['comparison'])
shap.plots.waterfall(out['_raw']['exp'][0], max_display=12)

In [ ]:
pd.DataFrame(out['lime']['rules'])

## 7. Équité

In [ ]:
audit = X.loc[Xte.index, D.SENSITIVE].copy(); audit['age'] = EV.age_band(audit['age'])
for c in audit:
    gm = EV.group_metrics(yte, proba, thr, audit[c].to_numpy()); print(gm.round(3), EV.fairness_gaps(gm), '\n')